# 02 — An Open-Economy CGE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/02_open_economy_cge.ipynb)

**Model:** Hosoe's `stdcge`.

Compared with Notebook 01, the economy now has government, taxes, imports, exports, intermediate inputs, saving, and investment.

Our experiment: **change an import tariff and let the entire economy re-equilibrate.**

## 1. Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# In Colab, set the environment variable CGE_CORE_REF to test a branch or tag.
# The public notebooks default to main. Outside Colab, a current CGE-Core git
# checkout is used directly, so branch development never silently resets to main.
CGE_CORE_REF = os.environ.get("CGE_CORE_REF", "main")
IN_COLAB = Path("/content").exists()

if not IN_COLAB and (Path.cwd() / ".git").is_dir() and (Path.cwd() / "cge_core").is_dir():
    REPO_DIR = Path.cwd()
    source_label = "current checkout"
else:
    WORKSPACE = Path("/content") if IN_COLAB else Path.home() / ".cache"
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = WORKSPACE / "CGE-core-colab"
    REPO_URL = "https://github.com/miraflor/CGE-core.git"

    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout.")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--no-checkout", REPO_URL, str(REPO_DIR)],
            check=True,
        )

    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", CGE_CORE_REF, "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
        check=True,
    )
    source_label = CGE_CORE_REF

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("✓ CGE-Core", cge_core.__version__)
print("✓ Source:", source_label, f"({commit})")
print("✓ Repository:", REPO_DIR)


import shutil

if not shutil.which("ipopt"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "amplpy.modules", "install", "coin"],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    module_path = subprocess.check_output(
        [sys.executable, "-m", "amplpy.modules", "path"],
        text=True,
    ).strip()
    os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)


## 2. Inspect the benchmark SAM

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from cge_core import CGE, example_data
from cge_core.models import StdCGE

data_dir = example_data("stdcge")
sam = pd.read_csv(data_dir / "param-sam-.csv", index_col=0)
GOODS = ["BRD", "MLK"]
FACTORS = ["CAP", "LAB"]
display(sam)

## 3. Solve the open-economy benchmark

In [ ]:
model = CGE(model=StdCGE(), data=data_dir)
benchmark = model.solve_benchmark(
    numeraire=("pf", "LAB"),
    redundant=("eqpf", "LAB"),
    solver=SOLVER,
)

print("Goods:", GOODS)
print("Factors:", FACTORS)
print("Benchmark welfare objective:", benchmark.objective)

print("\nBenchmark tariff rates:")
for good in GOODS:
    print(f"  {good}: {benchmark.value('taum', good):.2%}")

## 4. Your policy lever 👇

In [ ]:
# 👇 EDIT THESE
GOOD = "MLK"
NEW_TARIFF = 0.00

## 5. Solve the policy scenario

In [ ]:
if GOOD not in GOODS:
    raise ValueError(f"GOOD must be one of {GOODS}")

old_tariff = benchmark.value("taum", GOOD)
scenario = benchmark.scenario(f"{GOOD} tariff → {NEW_TARIFF:.2%}")
scenario.set("taum", GOOD, float(NEW_TARIFF))
result = scenario.solve(solver=SOLVER)
results = result.compare(benchmark)

print(f"{GOOD} tariff: {old_tariff:.2%} → {NEW_TARIFF:.2%}")
print(f"Welfare objective: {benchmark.objective:.4f} → {result.objective:.4f}")

## 6. Compare the economy before and after

In [ ]:
labels = {
    "Z": "Gross output",
    "M": "Imports",
    "E": "Exports",
    "Xp": "Household demand",
    "pq": "Composite price",
}
headline = results[results["component"].isin(labels)].copy()
headline["measure"] = headline["component"].map(labels)
headline["item"] = headline["index_1"].fillna("")
display(
    headline[["measure", "item", "reference_value", "value", "difference", "pct_change"]]
    .sort_values(["measure", "item"])
    .style.format({
        "reference_value": "{:.4f}",
        "value": "{:.4f}",
        "difference": "{:+.4f}",
        "pct_change": "{:+.2f}%",
    })
)

In [ ]:
for component, title in [
    ("Z", "Gross output"),
    ("M", "Imports"),
    ("E", "Exports"),
    ("Xp", "Household demand"),
    ("pq", "Composite prices"),
]:
    part = results[results["component"] == component]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(part["index_1"].astype(str), part["pct_change"].astype(float))
    ax.axhline(0, linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel("% change from benchmark")
    plt.show()

## 7. Hicksian equivalent variation

In [ ]:
def equivalent_variation(benchmark, result, goods):
    denom = math.prod(
        benchmark.value("alpha", good) ** benchmark.value("alpha", good)
        for good in goods
    )
    return result.objective / denom - benchmark.objective / denom

print(f"Equivalent variation: {equivalent_variation(benchmark, result, GOODS):+.4f}")

## What changed conceptually?

The tariff changes an import price wedge. That propagates through substitution between imports and domestic goods, production, exports, factor demand, income, government revenue, saving, household demand, and finally the new general equilibrium.

## Next

Notebook 03 turns this into a small policy laboratory.

[Open Notebook 03 in Colab](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/03_policy_experiments.ipynb)